In [2]:
from xarray_utils import analyze_netcdf, zarr_to_netcdf, find_missing_days
import pandas as pd
import xarray as xr
import numpy as np
import sklearn as sk
import sklearn as sk

In [3]:
# Open the Zarr dataset
ds_imd = xr.open_zarr("../data/raw/IMD_rainfall_0p25.zarr")
ds_imd = ds_imd.where(ds_imd != -999)

analyze_netcdf("../data/raw/IMD_rainfall_0p25.nc")

Analysis for NetCDF File: IMD_rainfall_0p25.nc

--- Dimensions ---
time: 31046
lat: 129
lon: 135

--- Coordinates ---
- lat:
    dtype: float64
    shape: (129,)
    attributes: {'axis': 'Y', 'long_name': 'latitude', 'standard_name': 'latitude', 'units': 'degrees_north'}
- lon:
    dtype: float64
    shape: (135,)
    attributes: {'axis': 'X', 'long_name': 'longitude', 'standard_name': 'longitude', 'units': 'degrees_east'}
- time:
    dtype: datetime64[ns]
    shape: (31046,)
    attributes: {'long_name': 'time', 'standard_name': 'time'}

--- Data Variables ---
- rain:
    dtype: float64
    shape: (31046, 129, 135)
    dimensions: ('time', 'lat', 'lon')
    attributes: {'long_name': 'Rainfall', 'units': 'mm/day'}

--- Global Attributes ---
Conventions: CF-1.7
comment: 
crs: epsg:4326
history: 2026-06-19 06:45:25.930728 Python
references: 
source: https://imdpune.gov.in/
title: IMD gridded data



In [4]:
# Open the Zarr dataset
ds_ecm = xr.open_zarr("../data/processed/s2s_reforecast_sorted.zarr")

analyze_netcdf("../data/raw/s2s_reforecast.nc")

Analysis for NetCDF File: s2s_reforecast.nc

--- Dimensions ---
time: 3720
step: 43
lat: 33
lon: 35

--- Coordinates ---
- lat:
    dtype: float64
    shape: (33,)
    attributes: {'long_name': 'latitude', 'standard_name': 'latitude', 'stored_direction': 'decreasing', 'units': 'degrees_north'}
- lon:
    dtype: float64
    shape: (35,)
    attributes: {'long_name': 'longitude', 'standard_name': 'longitude', 'units': 'degrees_east'}
- step:
    dtype: int64
    shape: (43,)
- time:
    dtype: datetime64[ns]
    shape: (3720,)
    attributes: {'long_name': 'initial time of forecast', 'standard_name': 'forecast_reference_time'}

--- Data Variables ---
- 10m_u_component_of_wind:
    dtype: float32
    shape: (3720, 43, 33, 35)
    dimensions: ('time', 'step', 'lat', 'lon')
    attributes: {'GRIB_NV': np.int64(0), 'GRIB_Nx': np.int64(35), 'GRIB_Ny': np.int64(33), 'GRIB_cfName': 'eastward_wind', 'GRIB_cfVarName': 'u10', 'GRIB_dataType': 'cf', 'GRIB_gridDefinitionDescription': 'Latitude/longi

In [5]:

# convert lead days into timedeltas
step_td = pd.to_timedelta(ds_ecm.step.values, unit="D").to_numpy()

ds_ecmv = ds_ecm.assign_coords(
    valid_time=(("time", "step"),
                ds_ecm.time.values[:, None] + step_td[None, :])
)

In [6]:
#all libraries required
import time
from contextlib import contextmanager

import numpy as np
import xarray as xr
from dask.diagnostics import ProgressBar

In [7]:
# ---- CONFIG ----
IMD_TARGET_VAR = "rain" 
CLIM_WINDOW_DAYS = 1
STEP_UNITS = "auto"
RIDGE = 1e-4            # on the standardised normal equations
CELL_CHUNK = 256        # cells per solve block -- controls peak RAM, not speed
DTYPE = np.float32

# Subsetting knobs -- these control whether this fits in RAM.
LEADS = None            # e.g. range(14, 43) for your target window; None = all
MONTHS = None           # e.g. (6, 7, 8, 9) for JJAS only; None = all year
COARSE_PAD = 4.0        # degrees of ECMWF kept beyond IMD's box (for interp edges)

OUT_SKILL_MAPS = "../results/models/skill_maps.nc"

@contextmanager
def stage(name):
    print(f"[ ] {name} ...", flush=True)
    t0 = time.perf_counter()
    yield    #everything before yield runs at entry, after runs at exit
    print(f"[x] {name}  ({time.perf_counter() - t0:.1f}s)", flush=True)

In [19]:
#ensure step dtype and valid_time 

def normalize_step(ds, units=STEP_UNITS, verbose=True):
    v = ds["step"].values
    if np.issubdtype(v.dtype, np.timedelta64):
        return ds
    if not np.issubdtype(v.dtype, np.number):
        raise TypeError(f"'step' has dtype {v.dtype}; expected timedelta64 or numeric.")
    u = units
    if u == "auto":
        mx = int(np.nanmax(v))
        u = "D" if mx <= 60 else ("h" if mx <= 24 * 60 else "s")
        if verbose:
            print(f"    step is {v.dtype} (max {mx}) -> reading as units='{u}'")
    td = v.astype(np.int64).astype(f"timedelta64[{u}]").astype("timedelta64[ns]")
    return ds.assign_coords(step=("step", td))

def ensure_valid_time(ds, verbose=True):
    ds = normalize_step(ds, verbose=verbose)
    expected = ds["time"] + ds["step"]
    if "valid_time" in ds.coords:
        got = ds["valid_time"]
        if got.shape == expected.shape and (got.values == expected.values).all():
            return ds
        if verbose:
            print("    WARNING: existing valid_time != time + step -> rebuilding")
        ds = ds.drop_vars("valid_time")
    return ds.assign_coords(valid_time=expected)

def step_days(ds):
    return (ds["step"].values / np.timedelta64(1, "D")).astype(int)

In [20]:
#the build cell function will remove the cells if no non-NaN values found for a lat-lon pair
def build_cells(imd_ds, var=IMD_TARGET_VAR):
    """Flatten IMD to a 1-D list of VALID cells only.

    Everything downstream works on (time, cell) instead of (time, lat, lon),
    which drops 71.5% of the grid before any arithmetic happens. The lat/lon
    of each cell are kept so we can interpolate ECMWF straight to these
    points and so we can unstack skill back into a map at the end.
    """
    da = imd_ds[var]
    mask2d = da.notnull().any(dim="time").compute()
    stacked = mask2d.stack(cell=("lat", "lon"))
    sel = stacked.values
    cells = stacked[sel]
    n, tot = int(sel.sum()), int(sel.size)
    print(f"    {n}/{tot} valid cells ({100 * n / tot:.1f}%) -- "
          f"interpolating straight to these, skipping the rest")
    return cells, mask2d

def cell_points(cells):
    lat = xr.DataArray(cells["lat"].values, dims="cell")
    lon = xr.DataArray(cells["lon"].values, dims="cell")
    return lat, lon 

In [21]:
#load both datasets time and define the slices or gaps between the points
def load_coarse(ecmwf_ds, imd_ds, months=MONTHS, leads=LEADS, pad=COARSE_PAD):
    """ECMWF over India's box, still at native coarse res, fully in RAM.

    This is the ONE disk pass. Keeping it coarse is what makes it small:
    the 0.25 expansion happens per lead, in memory, and is discarded.
    """
    ecmwf_ds = ensure_valid_time(ecmwf_ds)

    # lat may be descending (ECMWF convention); slice() silently returns
    # empty on a descending index, so normalise first.
    for c in ("lat", "lon"):
        if ecmwf_ds[c].values[0] > ecmwf_ds[c].values[-1]:
            ecmwf_ds = ecmwf_ds.sortby(c)

    lat0, lat1 = float(imd_ds.lat.min()), float(imd_ds.lat.max())
    lon0, lon1 = float(imd_ds.lon.min()), float(imd_ds.lon.max())
    sub = ecmwf_ds.sel(lat=slice(lat0 - pad, lat1 + pad),
                       lon=slice(lon0 - pad, lon1 + pad))

    if months is not None:
        sub = sub.sel(time=sub["time"].dt.month.isin(list(months)))
    if leads is not None:
        sub = sub.isel(step=list(leads))

    nbytes = np.prod([sub.sizes[d] for d in ("time", "step", "lat", "lon")]) \
        * len(sub.data_vars) * 4
    print(f"    coarse subset {dict(sub.sizes)} x {len(sub.data_vars)} vars "
          f"-> {nbytes / 1e9:.2f} GB in RAM")
    if nbytes > 8e9:
        print("    WARNING: >8 GB. Set MONTHS=(6,7,8,9) and/or LEADS=range(14,43).")

    with ProgressBar():
        sub = sub.astype(DTYPE).compute()
    return sub


def load_target(imd_ds, cells, var=IMD_TARGET_VAR):
    """IMD as (time, cell), valid cells only. ~KBs per day."""
    da = imd_ds[var].stack(cell=("lat", "lon"))
    da = da.sel(cell=cells["cell"])
    with ProgressBar():
        da = da.astype(DTYPE).compute()
    return da.assign_coords(time=da["time"].dt.floor("D"))


def lead_slice(coarse, j, lat_pts, lon_pts, feature_vars):
    """Interpolate ONE lead's coarse fields to the 4964 valid points.

    Pointwise interp (lat/lon as DataArrays sharing dim 'cell') means the
    129x135 rectangle is never allocated -- we only ever touch valid cells.
    Returns (n_time, n_cell, n_feat) float32, ~1.1 GB for train.
    """
    s = coarse.isel(step=j)
    fine = s[feature_vars].interp(lat=lat_pts, lon=lon_pts)
    return np.stack([fine[v].values for v in feature_vars], axis=-1).astype(DTYPE)


In [22]:
# calculating climatology

def _doy_matrix(doys, window, n_doy=366):
    """(n_doy, n_samples) circular-window membership matrix."""
    centers = np.arange(1, n_doy + 1)  #1-366 array, target_doys
    d = np.abs(doys[None, :].astype(int) - centers[:, None]) # distance from each sample to center
    return (np.minimum(d, n_doy - d) <= window).astype(DTYPE) #returns array for each sample, with true if sample is in window 


def _clim_grid(values, doys, window):
    """values (n, c) -> (n_doy, c) DOY climatology per cell, NaN-aware.

    The NaN handling is a matmul trick: counts = M @ isfinite gives the
    per-(doy, cell) valid sample count without ever building an (n_doy, n, c)
    boolean array (which would be ~5e9 elements here).
    """
    M = _doy_matrix(doys, window)
    finite = np.isfinite(values).astype(DTYPE)
    counts = M @ finite
    sums = M @ np.nan_to_num(values).astype(DTYPE)
    with np.errstate(invalid="ignore", divide="ignore"):
        clim = np.where(counts > 0, sums / np.maximum(counts, 1), np.nan)
    return clim.astype(DTYPE), counts

In [23]:
def fit_predict_cells(Xtr, ytr, Xev, ridge=RIDGE, chunk=CELL_CHUNK):
    """One regression per cell, chunked over cells.

    Xtr (n, c, f), ytr (n, c), Xev (m, c, f) -> pred (m, c)

    CHUNKING IS ABOUT MEMORY, NOT SPEED. Benchmarked at 4964 cells / 2604
    samples / 21 features, this is 6.7s vs 6.8s for an sklearn loop -- i.e.
    the same. What it buys is peak RSS: the unchunked version makes four
    full 1.1 GB copies (subtract, divide, nan_to_num, transpose) before it
    reaches BLAS and gets OOM-killed on a 16 GB machine. Chunked peaks ~3.4 GB.

    Standardise per cell before the solve. This is load-bearing, not hygiene:
    msl ~1e5 Pa and specific humidity ~1e-2 kg/kg differ by seven orders of
    magnitude, and float32 normal equations on that are singular in practice.
    After centering, no intercept column is needed.
    """
    n, c, f = Xtr.shape
    m = Xev.shape[0]
    out = np.empty((m, c), DTYPE)

    for a in range(0, c, chunk):
        b = min(a + chunk, c)
        xt, xe, yt = Xtr[:, a:b], Xev[:, a:b], ytr[:, a:b]

        mu = np.nanmean(xt, axis=0)
        sd = np.nanstd(xt, axis=0)
        sd = np.where(sd < 1e-12, 1.0, sd)      # dead predictors -> no-op
        ymu = np.nanmean(yt, axis=0)

        xs = np.nan_to_num((xt - mu) / sd)
        ys = np.nan_to_num(yt - ymu)

        XtX = np.einsum("ncf,ncg->cfg", xs, xs, optimize=True)
        Xty = np.einsum("ncf,nc->cf", xs, ys, optimize=True)
        XtX += (ridge * n) * np.eye(f, dtype=XtX.dtype)

        beta = np.linalg.solve(XtX, Xty[:, :, None])[:, :, 0]
        out[:, a:b] = np.einsum(
            "mcf,cf->mc", np.nan_to_num((xe - mu) / sd), beta) + ymu

    return out


def _rmse_cells(y, p):
    """Per-cell RMSE ignoring NaN. y, p: (n, c) -> (c,)"""
    e2 = (y - p) ** 2
    with np.errstate(invalid="ignore"):
        return np.sqrt(np.nanmean(e2, axis=0))


In [24]:
ecmwf_ds = ds_ecmv   # noqa: F821
imd_ds = ds_imd      # noqa: F821

t0 = time.perf_counter()

with stage("Building valid-cell index"):
    cells, mask2d = build_cells(imd_ds)
    lat_pts, lon_pts = cell_points(cells)

with stage("Loading coarse ECMWF into RAM (the only disk pass)"):
    coarse = load_coarse(ecmwf_ds, imd_ds)
    feature_vars = list(coarse.data_vars)
    leads = step_days(coarse)

with stage("Loading IMD target"):
    imd_cells = load_target(imd_ds, cells)

with stage("Splitting by init year"):
    yrs = sorted(set(coarse["time"].dt.year.values.tolist()))
    test_years, val_years = set(yrs[-3:]), set(yrs[-6:-3])
    y_of = coarse["time"].dt.year
    tr_i = ~(y_of.isin(list(test_years)) | y_of.isin(list(val_years))).values
    va_i = y_of.isin(list(val_years)).values
    print(f"    train {tr_i.sum()} | val {va_i.sum()} inits, "
            f"{len(feature_vars)} features, {len(leads)} leads")

n_cell = len(cells["cell"])
rows = []
maps = {"skill_anom": [], "skill_raw": [], "rmse_clim": []}

with stage(f"Fitting {n_cell} cells x {len(leads)} leads"):
    for j, lead in enumerate(leads):
        Xall = lead_slice(coarse, j, lat_pts, lon_pts, feature_vars)

        vt = coarse["valid_time"].isel(step=j).dt.floor("D").values
        yall = imd_cells.reindex(time=vt).values                 # (n, c)
        doy = xr.DataArray(vt, dims="t").dt.dayofyear.values

        Xtr, ytr, dtr = Xall[tr_i], yall[tr_i], doy[tr_i]
        Xev, yev, dev = Xall[va_i], yall[va_i], doy[va_i]

        # --- climatologies, train only, this lead only (~150 MB) ---
        clim_o, _ = _clim_grid(ytr, dtr, CLIM_WINDOW_DAYS)       # (366, c)
        clim_m = np.empty((366, n_cell, len(feature_vars)), DTYPE)
        for k in range(len(feature_vars)):
            clim_m[:, :, k], _ = _clim_grid(Xtr[:, :, k], dtr, CLIM_WINDOW_DAYS)

        # --- arm 1: per-cell DOY climatology (the reference) ---
        p_clim = clim_o[dev - 1]
        rmse_clim = _rmse_cells(yev, p_clim)

        # --- arm 2: raw ---
        p_raw = fit_predict_cells(Xtr, ytr, Xev)
        rmse_raw = _rmse_cells(yev, p_raw)

        # --- arm 3: anomaly (each side minus its OWN climatology;
        #     model clim is lead-dependent, obs clim is not) ---
        Xtr_a = Xtr - clim_m[dtr - 1]
        Xev_a = Xev - clim_m[dev - 1]
        ytr_a = ytr - clim_o[dtr - 1]
        yev_a = yev - clim_o[dev - 1]
        p_anom = fit_predict_cells(Xtr_a, ytr_a, Xev_a)
        rmse_anom = _rmse_cells(yev_a, p_anom)
        rmse_clim_a = _rmse_cells(yev_a, np.zeros_like(yev_a))

        with np.errstate(invalid="ignore", divide="ignore"):
            sk_raw = 1 - rmse_raw / rmse_clim
            sk_anom = 1 - rmse_anom / rmse_clim_a

        maps["skill_anom"].append(sk_anom)
        maps["skill_raw"].append(sk_raw)
        maps["rmse_clim"].append(rmse_clim)
        rows.append((int(lead), np.nanmean(rmse_clim), np.nanmean(rmse_raw),
                        np.nanmean(sk_raw), np.nanmean(rmse_anom),
                        np.nanmean(sk_anom), float(np.nanmean(sk_anom > 0))))

        del Xall, Xtr, Xev, clim_m
        print(f"    lead {int(lead):>2}: mean skill_anom {np.nanmean(sk_anom):+.3f} "
                f"| {100 * np.nanmean(sk_anom > 0):.0f}% of cells positive", flush=True)

with stage("Writing skill maps"):
    out = xr.Dataset(
        {k: (("lead", "cell"), np.stack(v)) for k, v in maps.items()},
        coords={"lead": leads, "cell": cells["cell"]},
    ).unstack("cell")
    out.to_netcdf(OUT_SKILL_MAPS)
    print(f"    -> {OUT_SKILL_MAPS}  {dict(out.sizes)}")

print(f"\nTotal: {time.perf_counter() - t0:.1f}s\n")
print("Domain-MEAN skill by lead (maps are in the netcdf; means hide "
        "regional structure -- plot them).\n")
print(f"{'lead':>4} {'rmse_clim':>9} {'rmse_raw':>9} {'skl_raw':>8} "
        f"{'rmse_anom':>9} {'skl_anom':>8} {'%cells+':>8}")
for r in rows:
    print(f"{r[0]:>4} {r[1]:>9.3f} {r[2]:>9.3f} {r[3]:>8.3f} "
            f"{r[4]:>9.3f} {r[5]:>8.3f} {100 * r[6]:>7.0f}%")

[ ] Building valid-cell index ...
    4964/17415 valid cells (28.5%) -- interpolating straight to these, skipping the rest
[x] Building valid-cell index  (1.4s)
[ ] Loading coarse ECMWF into RAM (the only disk pass) ...
    step is int64 (max 42) -> reading as units='D'
    coarse subset {'time': 3720, 'step': 43, 'lat': 33, 'lon': 35} x 21 vars -> 15.52 GB in RAM
[########################################] | 100% Completed | 30.50 ss
[x] Loading coarse ECMWF into RAM (the only disk pass)  (30.9s)
[ ] Loading IMD target ...
[########################################] | 100% Completed | 1.70 sms
[x] Loading IMD target  (3.2s)
[ ] Splitting by init year ...
    train 2604 | val 558 inits, 21 features, 43 leads
[x] Splitting by init year  (0.0s)
[ ] Fitting 4964 cells x 43 leads ...


/var/folders/h4/hvzvbp993dx41cy379v1lblm0000gn/T/ipykernel_67368/2720810981.py:25: RuntimeWarning: Mean of empty slice
  mu = np.nanmean(xt, axis=0)
/Users/abhimanyu/Desktop/s2s/stsenv/lib/python3.14/site-packages/numpy/lib/_nanfunctions_impl.py:1997: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,


    lead  0: mean skill_anom +0.069 | 94% of cells positive
    lead  1: mean skill_anom +0.154 | 97% of cells positive
    lead  2: mean skill_anom +0.113 | 93% of cells positive
    lead  3: mean skill_anom +0.094 | 95% of cells positive
    lead  4: mean skill_anom +0.071 | 91% of cells positive
    lead  5: mean skill_anom +0.063 | 93% of cells positive
    lead  6: mean skill_anom +0.048 | 89% of cells positive
    lead  7: mean skill_anom +0.044 | 91% of cells positive
    lead  8: mean skill_anom +0.036 | 91% of cells positive
    lead  9: mean skill_anom +0.028 | 87% of cells positive
    lead 10: mean skill_anom +0.020 | 85% of cells positive
    lead 11: mean skill_anom +0.015 | 81% of cells positive
    lead 12: mean skill_anom +0.007 | 66% of cells positive
    lead 13: mean skill_anom +0.003 | 59% of cells positive
    lead 14: mean skill_anom +0.001 | 55% of cells positive
    lead 15: mean skill_anom +0.002 | 57% of cells positive
    lead 16: mean skill_anom -0.000 | 53

In [8]:
"""
UNet downscaling: ECMWF S2S (coarse, windowed, anomaly space) -> IMD 0.25
rain anomaly over India, for one target window (default weeks 3-4).

This pulls together every element settled in the diagnostics phase:

  WINDOWED TARGET     week 3-4 mean, not daily leads. The correlation
                      diagnostics showed daily signal dies by lead ~22 but
                      windowed r roughly quadruples; days 15-28 is where the
                      contribution lives.
  ANOMALY SPACE       obs minus DOY climatology; predictors minus their own
                      window climatology (the drift correction). Train-year
                      climatology only -- no leakage.
  MASKED LOSS         supervision only on IMD's valid cells. Strict mask
                      (.all, 4885 cells) so every cell has a full record.
  MASK AS CHANNEL     plus normalised lat/lon, so the net knows where
                      supervision lives and where it is.
  COARSE INPUT        predictors stay at 1.5 deg; the coarse->fine expansion
                      happens INSIDE the model (precomputed grid_sample =
                      exact coordinate-aware bilinear, then conv refinement).
                      Nothing is pre-interpolated to 0.25 on disk.
  GROUPNORM           not BatchNorm: batch stats over a 72%-ocean domain
                      would be dominated by unsupervised pixels.

MEMORY BUDGET (M4 Air, 16 GB)
-----------------------------
Windowing is what makes this fit. One sample per init (not per lead):
    X:  3720 x 21 x ~26 x ~26  float32  ~ 0.2 GB   (coarse, in RAM)
    y:  3720 x 129 x 135       float32  ~ 0.26 GB  (fine, in RAM)
Whole dataset lives in memory; the DataLoader just indexes tensors.
Model ~2-6M params; batch 16 at 129x135 is comfortably under 2 GB peak.
The one expensive pass (window aggregation over the archive) runs once in
`prepare` and is cached to .npz -- afterwards every training run starts in
seconds.

USAGE
-----
  python s2s_unet.py prepare      # archive -> cached arrays (slow, once)
  python s2s_unet.py train        # train + evaluate + write skill maps
  python s2s_unet.py train --epochs 80 --batch 8 --base 32
"""

import os
import time
from contextlib import contextmanager
from types import SimpleNamespace

import numpy as np

# ---- CONFIG ----
IMD_TARGET_VAR = "rain"
WINDOW = (15, 28)               # week 3-4; (29, 42) for weeks 5-6
CLIM_WINDOW_DAYS = 7
COARSE_PAD = 3.0                # deg beyond IMD box kept in the input
MONTHS = None                   # e.g. (6, 7, 8, 9) for JJAS inits
VAL_YEARS_N, TEST_YEARS_N = 3, 3
CACHE = "../data/cache/unet_cache_w{lo}_{hi}.npz"
OUT_MAPS = "../results/models/unet_skill_maps_w{lo}_{hi}.nc"
DTYPE = np.float32


@contextmanager
def stage(name):
    print(f"[ ] {name} ...", flush=True)
    t0 = time.perf_counter()
    yield
    print(f"[x] {name}  ({time.perf_counter() - t0:.1f}s)", flush=True)


In [9]:


# ======================================================================
# PREPARE: archive -> cached arrays (numpy/xarray only, no torch needed)
# ======================================================================

def normalize_step(ds, verbose=True):
    v = ds["step"].values
    if np.issubdtype(v.dtype, np.timedelta64):
        return ds
    mx = int(np.nanmax(v))
    u = "D" if mx <= 60 else ("h" if mx <= 24 * 60 else "s")
    if verbose:
        print(f"    step is {v.dtype} (max {mx}) -> units='{u}'")
    td = v.astype(np.int64).astype(f"timedelta64[{u}]").astype("timedelta64[ns]")
    return ds.assign_coords(step=("step", td))


def ensure_valid_time(ds):
    ds = normalize_step(ds)
    expected = ds["time"] + ds["step"]
    if "valid_time" in ds.coords:
        got = ds["valid_time"]
        if got.shape == expected.shape and (got.values == expected.values).all():
            return ds
        print("    WARNING: existing valid_time != time + step -> rebuilding")
        ds = ds.drop_vars("valid_time")
    return ds.assign_coords(valid_time=expected)


def _doy_matrix(doys, window, n_doy=366):
    centers = np.arange(1, n_doy + 1)
    d = np.abs(doys[None, :].astype(int) - centers[:, None])
    return (np.minimum(d, n_doy - d) <= window).astype(DTYPE)


def _clim_grid(values, doys, window):
    """(n, ...) -> (366, ...). NaN-aware DOY climatology via matmul."""
    shp = values.shape[1:]
    v2 = values.reshape(len(values), -1)
    M = _doy_matrix(doys, window)
    counts = M @ np.isfinite(v2).astype(DTYPE)
    sums = M @ np.nan_to_num(v2).astype(DTYPE)
    with np.errstate(invalid="ignore", divide="ignore"):
        clim = np.where(counts > 0, sums / np.maximum(counts, 1), np.nan)
    return clim.reshape((366,) + shp).astype(DTYPE)


In [10]:
def prepare(ecmwf_ds, imd_ds, lo, hi, cache_path):
    import xarray as xr
    from dask.diagnostics import ProgressBar

    with stage("Coarse subset over padded India box"):
        ecmwf_ds = ensure_valid_time(ecmwf_ds)
        for c in ("lat", "lon"):
            if ecmwf_ds[c].values[0] > ecmwf_ds[c].values[-1]:
                ecmwf_ds = ecmwf_ds.sortby(c)
        la0, la1 = float(imd_ds.lat.min()), float(imd_ds.lat.max())
        lo0, lo1 = float(imd_ds.lon.min()), float(imd_ds.lon.max())
        sub = ecmwf_ds.sel(lat=slice(la0 - COARSE_PAD, la1 + COARSE_PAD),
                           lon=slice(lo0 - COARSE_PAD, lo1 + COARSE_PAD))
        if MONTHS is not None:
            sub = sub.sel(time=sub["time"].dt.month.isin(list(MONTHS)))
        feature_vars = list(sub.data_vars)
        print(f"    {dict(sub.sizes)} x {len(feature_vars)} vars")

    with stage(f"Window aggregation days {lo}-{hi} (the one slow pass)"):
        leads = (sub["step"].values / np.timedelta64(1, "D")).astype(int)
        sel = np.where((leads >= lo) & (leads <= hi))[0]
        if len(sel) == 0:
            raise ValueError(f"no leads in [{lo},{hi}]")
        with ProgressBar():
            Xw = sub.isel(step=sel).mean(dim="step").compute()
        X = np.stack([Xw[v].values for v in feature_vars], axis=1).astype(DTYPE)
        # X: (n_init, n_var, n_clat, n_clon)
        clat, clon = Xw["lat"].values, Xw["lon"].values
        print(f"    X {X.shape}  {X.nbytes/1e9:.2f} GB")

    with stage("IMD windowed target on the fine grid"):
        vt = sub["valid_time"].isel(step=sel).dt.floor("D").values
        imd = imd_ds[IMD_TARGET_VAR]
        imd = imd.assign_coords(time=imd["time"].dt.floor("D"))
        flat = vt.ravel()
        with ProgressBar():
            y_all = imd.reindex(time=flat).astype(DTYPE).compute().values
        y = np.nanmean(y_all.reshape(vt.shape + y_all.shape[1:]), axis=1)
        flat_lat, flat_lon = imd["lat"].values, imd["lon"].values
        print(f"    y {y.shape}  {y.nbytes/1e9:.2f} GB")

    with stage("Mask (strict), splits, climatologies (train only)"):
        # strict mask: cell valid on EVERY day => uniform record length
        mask = np.isfinite(y).all(axis=0)
        print(f"    strict mask: {int(mask.sum())} cells "
              f"(loose would be {int(np.isfinite(y).any(axis=0).sum())})")

        init = sub["time"].values
        years = init.astype("datetime64[Y]").astype(int) + 1970
        uy = np.unique(years)
        test_y, val_y = set(uy[-TEST_YEARS_N:]), \
            set(uy[-(TEST_YEARS_N + VAL_YEARS_N):-TEST_YEARS_N])
        tr = ~np.isin(years, list(test_y | val_y))
        va = np.isin(years, list(val_y))
        te = np.isin(years, list(test_y))
        print(f"    train {tr.sum()} | val {va.sum()} | test {te.sum()} inits")

        centre = init + np.timedelta64((lo + hi) // 2, "D")
        import xarray as xr
        doy = xr.DataArray(centre, dims="t").dt.dayofyear.values

        clim_y = _clim_grid(y[tr], doy[tr], CLIM_WINDOW_DAYS)     # (366, H, W)
        clim_X = _clim_grid(X[tr], doy[tr], CLIM_WINDOW_DAYS)     # (366, V, h, w)

        y_anom = y - clim_y[doy - 1]
        X_anom = X - clim_X[doy - 1]

        # per-variable standardisation of predictors (train stats)
        xm = np.nanmean(X_anom[tr], axis=(0, 2, 3), keepdims=True)
        xs = np.nanstd(X_anom[tr], axis=(0, 2, 3), keepdims=True)
        xs = np.where(xs < 1e-8, 1.0, xs)
        X_anom = np.nan_to_num((X_anom - xm) / xs)

        # target scale (single scalar; masked cells only) for loss conditioning
        ys = float(np.nanstd(y_anom[tr][:, mask]))
        print(f"    target anomaly std (train, masked): {ys:.3f} mm/day")

    with stage(f"Caching -> {cache_path}"):
        np.savez_compressed(
            cache_path,
            X=X_anom, y=y_anom, mask=mask, doy=doy,
            tr=tr, va=va, te=te, y_std=ys,
            clim_y=clim_y, clat=clat, clon=clon,
            flat_lat=flat_lat, flat_lon=flat_lon,
            feature_vars=np.array(feature_vars),
            init=init.astype("datetime64[s]").astype(np.int64),
        )
        print(f"    {os.path.getsize(cache_path)/1e9:.2f} GB on disk")



In [11]:

ecmwf_ds = ds_ecmv   # noqa: F821  <- replace with your open datasets
imd_ds = ds_imd      # noqa: F821
lo, hi = WINDOW
cache_path = CACHE.format(lo=lo, hi=hi)
out_maps = OUT_MAPS.format(lo=lo, hi=hi)

prepare(ecmwf_ds, imd_ds, lo, hi, cache_path)

[ ] Coarse subset over padded India box ...
    step is int64 (max 42) -> units='D'
    {'time': 3720, 'step': 43, 'lat': 33, 'lon': 35} x 21 vars
[x] Coarse subset over padded India box  (0.0s)
[ ] Window aggregation days 15-28 (the one slow pass) ...
[########################################] | 100% Completed | 6.38 sms
    X (3720, 21, 33, 35)  0.36 GB
[x] Window aggregation days 15-28 (the one slow pass)  (6.8s)
[ ] IMD windowed target on the fine grid ...
[########################################] | 100% Completed | 2.32 sms
    y (3720, 129, 135)  0.26 GB
[x] IMD windowed target on the fine grid  (6.6s)
[ ] Mask (strict), splits, climatologies (train only) ...


/var/folders/h4/hvzvbp993dx41cy379v1lblm0000gn/T/ipykernel_18662/3624606649.py:38: RuntimeWarning: Mean of empty slice
  y = np.nanmean(y_all.reshape(vt.shape + y_all.shape[1:]), axis=1)


    strict mask: 4964 cells (loose would be 4964)
    train 2604 | val 558 | test 558 inits
    target anomaly std (train, masked): 3.988 mm/day
[x] Mask (strict), splits, climatologies (train only)  (1.3s)
[ ] Caching -> unet_cache_w15_28.npz ...
    0.41 GB on disk
[x] Caching -> unet_cache_w15_28.npz  (10.2s)


In [14]:
# ======================================================================
# MODEL (torch imported lazily so `prepare` works without it)
# ======================================================================

def build_model_and_train(args, cache_path, out_maps):
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torch.utils.data import DataLoader, TensorDataset

    dev = ("mps" if torch.backends.mps.is_available()
           else "cuda" if torch.cuda.is_available() else "cpu")
    print(f"    device: {dev}")

    z = np.load(cache_path, allow_pickle=False)
    X, y, mask = z["X"], z["y"], z["mask"]
    tr, va, te = z["tr"], z["va"], z["te"]
    clat, clon = z["clat"], z["clon"]
    flat_lat, flat_lon = z["flat_lat"], z["flat_lon"]
    H, W = len(flat_lat), len(flat_lon)

    # ---- exact coarse->fine alignment as a precomputed grid_sample grid ----
    # grid_sample wants normalised coords in [-1, 1] over the COARSE extent.
    # This is coordinate-aware bilinear: no assumption that the grids nest.
    gy = 2 * (flat_lat - clat[0]) / (clat[-1] - clat[0]) - 1
    gx = 2 * (flat_lon - clon[0]) / (clon[-1] - clon[0]) - 1
    gyy, gxx = np.meshgrid(gy, gx, indexing="ij")
    samp_grid = torch.tensor(
        np.stack([gxx, gyy], axis=-1), dtype=torch.float32
    ).unsqueeze(0)                                    # (1, H, W, 2)

    # static channels: mask + normalised lat/lon
    lat2 = (flat_lat[:, None] - flat_lat.mean()) / flat_lat.std()
    lon2 = (flat_lon[None, :] - flat_lon.mean()) / flat_lon.std()
    static = np.stack([mask.astype(DTYPE),
                       np.broadcast_to(lat2, (H, W)).astype(DTYPE),
                       np.broadcast_to(lon2, (H, W)).astype(DTYPE)])
    static_t = torch.tensor(static).unsqueeze(0)      # (1, 3, H, W)

    n_var = X.shape[1]

    def gn(c):
        return nn.GroupNorm(min(8, c), c)

    class Block(nn.Module):
        def __init__(self, ci, co):
            super().__init__()
            self.f = nn.Sequential(
                nn.Conv2d(ci, co, 3, padding=1), gn(co), nn.SiLU(),
                nn.Conv2d(co, co, 3, padding=1), gn(co), nn.SiLU())

        def forward(self, x):
            return self.f(x)

    class DownscaleUNet(nn.Module):
        """Coarse predictors in; fine anomaly field out.

        Coarse branch encodes synoptic context at native 1.5 deg. Its
        features are lifted to the fine grid by the precomputed grid_sample
        (exact coordinate mapping), concatenated with static channels, then
        refined by a small UNet at 0.25 deg. The upsampling is therefore
        learned-refined, never a stored interpolation.
        """
        def __init__(self, n_var, base=8):
            super().__init__()
            self.enc_c1 = Block(n_var, base * 2)
            self.enc_c2 = Block(base * 2, base * 2)          # coarse context
            self.inp = Block(base * 2 + 3, base)             # + static
            self.d1 = Block(base, base * 2)
            self.d2 = Block(base * 2, base * 4)
            self.bott = Block(base * 4, base * 4)
            self.u2 = nn.ConvTranspose2d(base * 4, base * 2, 2, stride=2)
            self.du2 = Block(base * 4, base * 2)
            self.u1 = nn.ConvTranspose2d(base * 2, base, 2, stride=2)
            self.du1 = Block(base * 2, base)
            self.head = nn.Conv2d(base, 1, 1)
            self.pool = nn.MaxPool2d(2)

        def forward(self, xc, samp, static):
            b = xc.shape[0]
            c = self.enc_c2(self.enc_c1(xc))                 # (b, 2base, h, w)
            f = F.grid_sample(c, samp.expand(b, -1, -1, -1),
                              mode="bilinear", align_corners=True)
            f = torch.cat([f, static.expand(b, -1, -1, -1)], dim=1)

            # pad fine grid to a multiple of 4 for two pool/unpool levels
            H0, W0 = f.shape[-2:]
            ph, pw = (-H0) % 4, (-W0) % 4
            f = F.pad(f, (0, pw, 0, ph), mode="replicate")

            e0 = self.inp(f)
            e1 = self.d1(self.pool(e0))
            e2 = self.d2(self.pool(e1))
            btm = self.bott(e2)
            u = self.du2(torch.cat([self.u2(btm), e1], dim=1))
            u = self.du1(torch.cat([self.u1(u), e0], dim=1))
            out = self.head(u)[:, :, :H0, :W0]
            return out.squeeze(1)

    # ---- data ----
    Xt = torch.tensor(X)
    yt = torch.tensor(np.nan_to_num(y))
    fin = torch.tensor(np.isfinite(y) & mask[None])          # per-sample mask
    mk_t = fin.float()

    def loader(idx, shuffle):
        ds = TensorDataset(Xt[idx], yt[idx], mk_t[idx])
        return DataLoader(ds, batch_size=args.batch, shuffle=shuffle,
                          num_workers=0)

    tr_dl = loader(np.where(tr)[0], True)
    va_dl = loader(np.where(va)[0], False)

    model = DownscaleUNet(n_var, base=args.base).to(dev)
    n_par = sum(p.numel() for p in model.parameters())
    print(f"    params: {n_par/1e6:.2f} M")
    samp = samp_grid.to(dev)
    stat = static_t.to(dev)
    opt = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=1e-3)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=args.epochs)

    def masked_mse(pred, target, m):
        # fill-then-mask-then-normalise-by-mask; never by numel
        se = (pred - target) ** 2 * m
        return se.sum() / m.sum().clamp(min=1.0)

    def run_epoch(dl, train):
        model.train(train)
        tot, cnt = 0.0, 0
        with torch.set_grad_enabled(train):
            for xb, yb, mb in dl:
                xb, yb, mb = xb.to(dev), yb.to(dev), mb.to(dev)
                pred = model(xb, samp, stat)
                loss = masked_mse(pred, yb, mb)
                if train:
                    opt.zero_grad(set_to_none=True)
                    loss.backward()
                    opt.step()
                tot += float(loss) * len(xb)
                cnt += len(xb)
        return tot / cnt

    with stage(f"Training {args.epochs} epochs, batch {args.batch}"):
        best, best_state, patience = np.inf, None, 0
        for ep in range(args.epochs):
            tl = run_epoch(tr_dl, True)
            vl = run_epoch(va_dl, False)
            sched.step()
            star = ""
            if vl < best - 1e-6:
                best, patience = vl, 0
                best_state = {k: v.detach().cpu().clone()
                              for k, v in model.state_dict().items()}
                star = "  *"
            else:
                patience += 1
            print(f"    ep {ep:>3}  train {tl:.4f}  val {vl:.4f}{star}", flush=True)
            if patience >= args.patience:
                print(f"    early stop (no val improvement in {args.patience})")
                break
        model.load_state_dict(best_state)

    # ---- evaluate: skill vs climatology (predict-zero-anomaly), per cell ----
    with stage("Evaluating on validation + test"):
        import xarray as xr
        results = {}
        for name, idx in (("val", np.where(va)[0]), ("test", np.where(te)[0])):
            preds = []
            model.eval()
            with torch.no_grad():
                for a in range(0, len(idx), args.batch):
                    xb = Xt[idx[a:a + args.batch]].to(dev)
                    preds.append(model(xb, samp, stat).cpu().numpy())
            p = np.concatenate(preds)                        # (n, H, W) anomaly
            t = y[idx]                                       # NaN outside mask

            fin_e = np.isfinite(t) & mask[None]
            se_m = np.where(fin_e, (t - p) ** 2, np.nan)
            se_c = np.where(fin_e, t ** 2, np.nan)           # clim = 0 anomaly
            with np.errstate(invalid="ignore"):
                rmse_m = np.sqrt(np.nanmean(se_m, axis=0))
                rmse_c = np.sqrt(np.nanmean(se_c, axis=0))
                skill = 1 - rmse_m / rmse_c
                # per-cell ACC
                tm = np.nanmean(np.where(fin_e, t, np.nan), axis=0)
                pm = np.nanmean(np.where(fin_e, p, np.nan), axis=0)
                num = np.nansum(np.where(fin_e, (t - tm) * (p - pm), np.nan), axis=0)
                den = np.sqrt(np.nansum(np.where(fin_e, (t - tm) ** 2, np.nan), axis=0)
                              * np.nansum(np.where(fin_e, (p - pm) ** 2, np.nan), axis=0))
                acc = np.where(den > 0, num / den, np.nan)
            results[name] = (skill, acc)
            valid_skill = np.isfinite(skill) & mask & (rmse_c > 1e-6)

            ms, ma = np.nanmean(skill[valid_skill]), np.nanmean(acc[mask])
            pos = 100 * np.nanmean(skill[mask] > 0)
            print(f"    {name}: mean skill {ms:+.3f} | mean ACC {ma:.3f} "
                  f"| {pos:.0f}% cells positive")

        out = xr.Dataset(
            {f"{n}_{k}": (("lat", "lon"), arr)
             for n, (s, a) in results.items()
             for k, arr in (("skill", s), ("acc", a))},
            coords={"lat": flat_lat, "lon": flat_lon},
        )
        out.attrs["window_days"] = f"{WINDOW[0]}-{WINDOW[1]}"
        out.attrs["note"] = ("skill = 1 - rmse/rmse_clim in anomaly space; "
                             "clim = predict zero anomaly. NaN outside IMD mask.")
        out.to_netcdf(out_maps)
        torch.save({"state": best_state, "args": vars(args)},
                   out_maps.replace(".nc", ".pt"))
        print(f"    -> {out_maps} (+ .pt checkpoint). Plot in Panoply directly.")


# ======================================================================
# Notebook execution block
# ======================================================================

RUN_MODE = "train" 
EPOCHS = 30
BATCH = 16
BASE =8
LR = 2e-4
PATIENCE = 10

args = SimpleNamespace(
    cmd=RUN_MODE,
    epochs=EPOCHS,
    batch=BATCH,
    base=BASE,
    lr=LR,
    patience=PATIENCE,
)

if RUN_MODE == "train":
    if not os.path.exists(cache_path):
        raise FileNotFoundError(f"run prepare first ({cache_path} missing)")
    build_model_and_train(args, cache_path, out_maps)
else:
    raise ValueError('RUN_MODE must be "prepare" or "train"')

    device: mps
    params: 0.06 M
[ ] Training 30 epochs, batch 16 ...
    ep   0  train 15.8275  val 19.4354  *
    ep   1  train 15.6692  val 19.3812  *
    ep   2  train 15.5936  val 19.3201  *
    ep   3  train 15.5506  val 19.2267  *
    ep   4  train 15.4921  val 19.2313
    ep   5  train 15.4410  val 19.1684  *
    ep   6  train 15.3886  val 19.1063  *
    ep   7  train 15.3462  val 19.1150
    ep   8  train 15.2866  val 19.2034
    ep   9  train 15.2426  val 19.3148
    ep  10  train 15.1711  val 19.0432  *
    ep  11  train 15.0883  val 19.2095
    ep  12  train 15.0282  val 19.0276  *
    ep  13  train 14.9494  val 19.1561
    ep  14  train 14.8688  val 19.2375
    ep  15  train 14.7860  val 19.1571
    ep  16  train 14.6972  val 19.1483
    ep  17  train 14.6182  val 19.2508
    ep  18  train 14.5476  val 19.2351
    ep  19  train 14.4654  val 19.2213
    ep  20  train 14.3973  val 19.2185
    ep  21  train 14.3360  val 19.2274
    ep  22  train 14.2815  val 19.2778
    ear

/var/folders/h4/hvzvbp993dx41cy379v1lblm0000gn/T/ipykernel_18662/3517383278.py:181: RuntimeWarning: Mean of empty slice
  rmse_m = np.sqrt(np.nanmean(se_m, axis=0))
/var/folders/h4/hvzvbp993dx41cy379v1lblm0000gn/T/ipykernel_18662/3517383278.py:182: RuntimeWarning: Mean of empty slice
  rmse_c = np.sqrt(np.nanmean(se_c, axis=0))
/var/folders/h4/hvzvbp993dx41cy379v1lblm0000gn/T/ipykernel_18662/3517383278.py:183: RuntimeWarning: divide by zero encountered in divide
  skill = 1 - rmse_m / rmse_c
/var/folders/h4/hvzvbp993dx41cy379v1lblm0000gn/T/ipykernel_18662/3517383278.py:185: RuntimeWarning: Mean of empty slice
  tm = np.nanmean(np.where(fin_e, t, np.nan), axis=0)
/var/folders/h4/hvzvbp993dx41cy379v1lblm0000gn/T/ipykernel_18662/3517383278.py:186: RuntimeWarning: Mean of empty slice
  pm = np.nanmean(np.where(fin_e, p, np.nan), axis=0)


    val: mean skill +0.008 | mean ACC 0.128 | 69% cells positive
    test: mean skill +0.006 | mean ACC 0.132 | 69% cells positive
    -> unet_skill_maps_w15_28.nc (+ .pt checkpoint). Plot in Panoply directly.
[x] Evaluating on validation + test  (1.2s)


In [ ]:
"""
Multi-window, lead-conditioned UNet downscaling: ECMWF S2S -> IMD 0.25 rain
anomaly over India, trained jointly across several DISJOINT forecast windows,
evaluated by rotating-year cross-validation.

WHY MULTI-WINDOW (the point of this rewrite)
--------------------------------------------
The single-window version reduced each init to one sample (3720 total) and
overfit hard -- and changing model size did nothing, which means the binding
constraint is SAMPLES, not capacity. Two facts drive the fix:

  * Adding all daily leads does NOT add real data. Day-20 and day-21 from one
    init share nearly identical predictor fields and nearly identical targets.
    43 daily leads from an init is ~3 effectively-independent samples, not 43.
    Worse, adjacent leads split across train/val is near-duplication = leak.

  * Adding DISJOINT windows DOES add data. Week-2, weeks 3-4 and weeks 5-6
    dynamics genuinely differ, and their valid-date ranges don't overlap, so
    no leak. One model correcting all three, told which window it is via an
    embedding channel, shares structure across leads -- the data-rich short
    leads regularise the data-poor long leads.

So: WINDOWS below are non-overlapping. Each (init, window) is one sample with
a window-id. Effective sample count ~3x. This is the lever that actually
moves val skill; base/dropout/weight-decay are second-order.

WHY ROTATING-YEAR CV
--------------------
20 years is few. A single 3-year val holdout is noisy -- one El Nino year in
val can dominate the estimate. Rotating leave-3-out CV over the non-test years
gives a stable mean +/- spread and uses every year for validation once.
Test years are held out of ALL folds, always.

Everything else carries over from the settled pipeline: anomaly space with
train-only climatology, strict mask, mask+lat/lon static channels,
coarse-input with in-model grid_sample upsampling, GroupNorm, masked loss.

USAGE
-----
  python s2s_unet_mw.py prepare        # archive -> cached arrays (slow, once)
  python s2s_unet_mw.py train          # rotating-year CV, then final model
  python s2s_unet_mw.py train --epochs 60 --batch 16 --base 24 --folds 5
"""

import argparse
import os
import time
from contextlib import contextmanager

import numpy as np

# ---- CONFIG ----
IMD_TARGET_VAR = "rain"

# DISJOINT windows: (name, first_lead_day, last_lead_day) inclusive.
# Non-overlap in valid dates is what keeps the extra samples honest.
WINDOWS = [
    ("week1", 1, 7),
    ("week2", 8, 14),
    ("week3", 15, 21),
    ("week4", 22, 28),
    ("week5", 29, 35),
    ("week6", 36, 42),
]

CLIM_WINDOW_DAYS = 7
COARSE_PAD = 3.0
MONTHS = None                   # e.g. (6, 7, 8, 9) for JJAS inits
TEST_YEARS_N = 3                # held out of every CV fold
CACHE = "../data/cache/unet_cache_mw.npz"
OUT_MAPS = "../results/models/unet_skill_maps_mw.nc"
DTYPE = np.float32


@contextmanager
def stage(name):
    print(f"[ ] {name} ...", flush=True)
    t0 = time.perf_counter()
    yield
    print(f"[x] {name}  ({time.perf_counter() - t0:.1f}s)", flush=True)


# ======================================================================
# PREPARE  (numpy/xarray only; torch not needed here)
# ======================================================================

def normalize_step(ds, verbose=True):
    v = ds["step"].values
    if np.issubdtype(v.dtype, np.timedelta64):
        return ds
    mx = int(np.nanmax(v))
    u = "D" if mx <= 60 else ("h" if mx <= 24 * 60 else "s")
    if verbose:
        print(f"    step is {v.dtype} (max {mx}) -> units='{u}'")
    td = v.astype(np.int64).astype(f"timedelta64[{u}]").astype("timedelta64[ns]")
    return ds.assign_coords(step=("step", td))


def ensure_valid_time(ds):
    ds = normalize_step(ds)
    expected = ds["time"] + ds["step"]
    if "valid_time" in ds.coords:
        got = ds["valid_time"]
        if got.shape == expected.shape and (got.values == expected.values).all():
            return ds
        print("    WARNING: existing valid_time != time + step -> rebuilding")
        ds = ds.drop_vars("valid_time")
    return ds.assign_coords(valid_time=expected)


def _doy_matrix(doys, window, n_doy=366):
    centers = np.arange(1, n_doy + 1)
    d = np.abs(doys[None, :].astype(int) - centers[:, None])
    return (np.minimum(d, n_doy - d) <= window).astype(DTYPE)


def _clim_grid(values, doys, window):
    """(n, ...) -> (366, ...), NaN-aware DOY climatology via matmul."""
    shp = values.shape[1:]
    v2 = values.reshape(len(values), -1)
    M = _doy_matrix(doys, window)
    counts = M @ np.isfinite(v2).astype(DTYPE)
    sums = M @ np.nan_to_num(v2).astype(DTYPE)
    with np.errstate(invalid="ignore", divide="ignore"):
        clim = np.where(counts > 0, sums / np.maximum(counts, 1), np.nan)
    return clim.reshape((366,) + shp).astype(DTYPE)


def prepare(ecmwf_ds, imd_ds, cache_path):
    import xarray as xr
    from dask.diagnostics import ProgressBar

    with stage("Coarse subset over padded India box"):
        ecmwf_ds = ensure_valid_time(ecmwf_ds)
        for c in ("lat", "lon"):
            if ecmwf_ds[c].values[0] > ecmwf_ds[c].values[-1]:
                ecmwf_ds = ecmwf_ds.sortby(c)
        la0, la1 = float(imd_ds.lat.min()), float(imd_ds.lat.max())
        lo0, lo1 = float(imd_ds.lon.min()), float(imd_ds.lon.max())
        sub = ecmwf_ds.sel(lat=slice(la0 - COARSE_PAD, la1 + COARSE_PAD),
                           lon=slice(lo0 - COARSE_PAD, lo1 + COARSE_PAD))
        if MONTHS is not None:
            sub = sub.sel(time=sub["time"].dt.month.isin(list(MONTHS)))
        feature_vars = list(sub.data_vars)
        clat, clon = sub["lat"].values, sub["lon"].values
        leads = (sub["step"].values / np.timedelta64(1, "D")).astype(int)
        init = sub["time"].values
        print(f"    {dict(sub.sizes)} x {len(feature_vars)} vars")

    imd = imd_ds[IMD_TARGET_VAR]
    imd = imd.assign_coords(time=imd["time"].dt.floor("D"))
    flat_lat, flat_lon = imd["lat"].values, imd["lon"].values

    # Build each window's (X, y, doy) then stack, tagging window id.
    X_list, y_list, doy_list, wid_list = [], [], [], []
    for wid, (wname, lo, hi) in enumerate(WINDOWS):
        with stage(f"Window {wname} (days {lo}-{hi})"):
            sel = np.where((leads >= lo) & (leads <= hi))[0]
            if len(sel) == 0:
                raise ValueError(f"no leads in [{lo},{hi}]")

            with ProgressBar():
                Xw = sub.isel(step=sel).mean(dim="step").compute()
            Xa = np.stack([Xw[v].values for v in feature_vars], axis=1).astype(DTYPE)

            vt = sub["valid_time"].isel(step=sel).dt.floor("D").values
            with ProgressBar():
                y_all = imd.reindex(time=vt.ravel()).astype(DTYPE).compute().values
            ya = np.nanmean(y_all.reshape(vt.shape + y_all.shape[1:]), axis=1)

            centre = init + np.timedelta64((lo + hi) // 2, "D")
            doya = xr.DataArray(centre, dims="t").dt.dayofyear.values

            X_list.append(Xa)
            y_list.append(ya)
            doy_list.append(doya)
            wid_list.append(np.full(len(Xa), wid, dtype=np.int64))
            print(f"    +{len(Xa)} samples")

    X = np.concatenate(X_list)          # (N, V, h, w)  N = n_init * n_window
    y = np.concatenate(y_list)          # (N, H, W)
    doy = np.concatenate(doy_list)      # (N,)
    wid = np.concatenate(wid_list)      # (N,)
    year = np.concatenate([init.astype("datetime64[Y]").astype(int) + 1970] * len(WINDOWS))
    print(f"\n    total {len(X)} samples ({len(init)} inits x {len(WINDOWS)} windows)")

    with stage("Strict mask + test-year holdout"):
        # strict mask: valid on every day, per window then intersect
        mask = np.isfinite(y).all(axis=0)
        print(f"    strict mask: {int(mask.sum())} cells "
              f"(loose {int(np.isfinite(y).any(axis=0).sum())})")
        uy = np.unique(year)
        test_years = set(uy[-TEST_YEARS_N:])
        is_test = np.isin(year, list(test_years))
        print(f"    test years (held out of all folds): {sorted(test_years)}")

    with stage("Caching"):
        # NOTE: climatology + standardisation are deferred to train time,
        # because they must be recomputed per CV FOLD (train-year stats only).
        # Caching raw windowed anomable inputs would bake in a fixed split.
        np.savez_compressed(
            cache_path,
            X=X, y=y, doy=doy, wid=wid, year=year,
            mask=mask, is_test=is_test,
            clat=clat, clon=clon, flat_lat=flat_lat, flat_lon=flat_lon,
            feature_vars=np.array(feature_vars),
            window_names=np.array([w[0] for w in WINDOWS]),
        )
        print(f"    {os.path.getsize(cache_path)/1e9:.2f} GB -> {cache_path}")
        print("    (raw windowed values cached; climatology/standardisation")
        print("     done per-fold at train time so each fold is leak-free)")


# ======================================================================
# FOLD-LOCAL PREPROCESSING  (must run per fold: train-year stats only)
# ======================================================================

def anomalise_fold(X, y, doy, tr):
    """De-climatologise both sides using TRAIN-fold years only, then
    standardise predictors on train stats. Returns processed copies.

    This is the leakage-critical step. Doing it once globally would let the
    val/test years inform the climatology; doing it per fold keeps each
    fold's estimate honest.
    """
    clim_y = _clim_grid(y[tr], doy[tr], CLIM_WINDOW_DAYS)      # (366, H, W)
    clim_X = _clim_grid(X[tr], doy[tr], CLIM_WINDOW_DAYS)      # (366, V, h, w)

    ya = y - clim_y[doy - 1]
    Xa = X - clim_X[doy - 1]

    xm = np.nanmean(Xa[tr], axis=(0, 2, 3), keepdims=True)
    xs = np.nanstd(Xa[tr], axis=(0, 2, 3), keepdims=True)
    xs = np.where(xs < 1e-8, 1.0, xs)
    Xa = np.nan_to_num((Xa - xm) / xs)
    return Xa.astype(DTYPE), ya.astype(DTYPE), clim_y


# ======================================================================
# MODEL + TRAIN  (torch, lazy import)
# ======================================================================

def build_and_run(args, cache_path, out_maps):
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torch.utils.data import DataLoader, TensorDataset

    dev = ("mps" if torch.backends.mps.is_available()
           else "cuda" if torch.cuda.is_available() else "cpu")
    print(f"    device: {dev}")

    z = np.load(cache_path, allow_pickle=False)
    X, y, doy, wid = z["X"], z["y"], z["doy"], z["wid"]
    year, mask, is_test = z["year"], z["mask"], z["is_test"]
    clat, clon = z["clat"], z["clon"]
    flat_lat, flat_lon = z["flat_lat"], z["flat_lon"]
    n_win = len(z["window_names"])
    H, W = len(flat_lat), len(flat_lon)
    n_var = X.shape[1]

    # ---- coarse->fine grid_sample grid (coordinate-aware bilinear) ----
    gy = 2 * (flat_lat - clat[0]) / (clat[-1] - clat[0]) - 1
    gx = 2 * (flat_lon - clon[0]) / (clon[-1] - clon[0]) - 1
    gyy, gxx = np.meshgrid(gy, gx, indexing="ij")
    samp_np = np.stack([gxx, gyy], -1).astype(np.float32)[None]
    assert np.abs(samp_np).max() <= 1.0, "fine grid outside coarse box; raise COARSE_PAD"

    lat2 = (flat_lat[:, None] - flat_lat.mean()) / flat_lat.std()
    lon2 = (flat_lon[None, :] - flat_lon.mean()) / flat_lon.std()
    static_np = np.stack([mask.astype(DTYPE),
                          np.broadcast_to(lat2, (H, W)).astype(DTYPE),
                          np.broadcast_to(lon2, (H, W)).astype(DTYPE)])[None]

    def gn(c):
        for g in (8, 4, 2, 1):
            if c % g == 0:
                return nn.GroupNorm(g, c)

    class Block(nn.Module):
        def __init__(self, ci, co, drop=0.0):
            super().__init__()
            layers = [nn.Conv2d(ci, co, 3, padding=1), gn(co), nn.SiLU(),
                      nn.Conv2d(co, co, 3, padding=1), gn(co), nn.SiLU()]
            if drop > 0:
                layers.append(nn.Dropout2d(drop))
            self.f = nn.Sequential(*layers)

        def forward(self, x):
            return self.f(x)

    class MWUNet(nn.Module):
        """Coarse predictors + window-id -> fine anomaly field.

        The window id is embedded and broadcast as an extra coarse-input
        channel, so one model corrects all windows and shares structure
        across leads. Everything else is the settled downscaling UNet:
        coarse encoder -> grid_sample to fine -> static channels -> small
        UNet refinement.
        """
        def __init__(self, n_var, n_win, base=24, drop=0.2, emb=4):
            super().__init__()
            self.emb = nn.Embedding(n_win, emb)
            self.enc_c1 = Block(n_var + emb, base * 2)
            self.enc_c2 = Block(base * 2, base * 2)
            self.inp = Block(base * 2 + 3, base)
            self.d1 = Block(base, base * 2, drop)
            self.d2 = Block(base * 2, base * 4, drop)
            self.bott = Block(base * 4, base * 4, drop)
            self.u2 = nn.ConvTranspose2d(base * 4, base * 2, 2, stride=2)
            self.du2 = Block(base * 4, base * 2, drop)
            self.u1 = nn.ConvTranspose2d(base * 2, base, 2, stride=2)
            self.du1 = Block(base * 2, base)
            self.head = nn.Conv2d(base, 1, 1)
            self.pool = nn.MaxPool2d(2)

        def forward(self, xc, wid, samp, static):
            b, _, h, w = xc.shape
            e = self.emb(wid)[:, :, None, None].expand(-1, -1, h, w)
            xc = torch.cat([xc, e], dim=1)
            c = self.enc_c2(self.enc_c1(xc))
            f = F.grid_sample(c, samp.expand(b, -1, -1, -1),
                              mode="bilinear", align_corners=True)
            f = torch.cat([f, static.expand(b, -1, -1, -1)], dim=1)
            H0, W0 = f.shape[-2:]
            f = F.pad(f, (0, (-W0) % 4, 0, (-H0) % 4), mode="replicate")
            e0 = self.inp(f)
            e1 = self.d1(self.pool(e0))
            e2 = self.d2(self.pool(e1))
            u = self.du2(torch.cat([self.u2(self.bott(e2)), e1], dim=1))
            u = self.du1(torch.cat([self.u1(u), e0], dim=1))
            return self.head(u)[:, :, :H0, :W0].squeeze(1)

    samp = torch.tensor(samp_np).to(dev)
    stat = torch.tensor(static_np).to(dev)

    def masked_mse(pred, target, m):
        se = (pred - target) ** 2 * m
        return se.sum() / m.sum().clamp(min=1.0)

    def skill_acc(p, t, fin):
        """Per-cell skill (vs zero-anomaly clim) and ACC over masked cells."""
        se_m = np.where(fin, (t - p) ** 2, np.nan)
        se_c = np.where(fin, t ** 2, np.nan)
        with np.errstate(invalid="ignore"):
            rmse_m = np.sqrt(np.nanmean(se_m, axis=0))
            rmse_c = np.sqrt(np.nanmean(se_c, axis=0))
            ok = rmse_c > 1e-6                       # guard degenerate cells (-inf fix)
            skill = np.where(ok, 1 - rmse_m / np.where(ok, rmse_c, 1), np.nan)
            tm = np.nanmean(np.where(fin, t, np.nan), axis=0)
            pm = np.nanmean(np.where(fin, p, np.nan), axis=0)
            num = np.nansum(np.where(fin, (t - tm) * (p - pm), np.nan), axis=0)
            den = np.sqrt(np.nansum(np.where(fin, (t - tm) ** 2, np.nan), axis=0)
                          * np.nansum(np.where(fin, (p - pm) ** 2, np.nan), axis=0))
            acc = np.where(den > 0, num / den, np.nan)
        return skill, acc

    def train_one(tr_idx, va_idx, Xa, ya, max_epochs, tag):
        Xt = torch.tensor(Xa)
        yt = torch.tensor(np.nan_to_num(ya))
        widt = torch.tensor(wid)
        fin = torch.tensor((np.isfinite(ya) & mask[None]).astype(DTYPE))

        def dl(idx, sh):
            return DataLoader(TensorDataset(Xt[idx], widt[idx], yt[idx], fin[idx]),
                              batch_size=args.batch, shuffle=sh)

        tr_dl, va_dl = dl(tr_idx, True), dl(va_idx, False)
        model = MWUNet(n_var, n_win, base=args.base, drop=args.drop).to(dev)
        opt = torch.optim.AdamW(model.parameters(), lr=args.lr,
                                weight_decay=args.wd)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max_epochs)

        best, best_state, wait = np.inf, None, 0
        for ep in range(max_epochs):
            model.train()
            for xb, wb, yb, mb in tr_dl:
                xb, wb, yb, mb = [t.to(dev) for t in (xb, wb, yb, mb)]
                loss = masked_mse(model(xb, wb, samp, stat), yb, mb)
                opt.zero_grad(set_to_none=True)
                loss.backward()
                opt.step()
            sched.step()

            model.eval()
            vl, n = 0.0, 0
            with torch.no_grad():
                for xb, wb, yb, mb in va_dl:
                    xb, wb, yb, mb = [t.to(dev) for t in (xb, wb, yb, mb)]
                    vl += float(masked_mse(model(xb, wb, samp, stat), yb, mb)) * len(xb)
                    n += len(xb)
            vl /= n
            if vl < best - 1e-6:
                best, wait = vl, 0
                best_state = {k: v.detach().cpu().clone()
                              for k, v in model.state_dict().items()}
            else:
                wait += 1
            if wait >= args.patience:
                break
        model.load_state_dict(best_state)
        return model, best, ep + 1

    def predict(model, idx, Xa):
        Xt = torch.tensor(Xa)
        widt = torch.tensor(wid)
        out = []
        model.eval()
        with torch.no_grad():
            for a in range(0, len(idx), args.batch):
                j = idx[a:a + args.batch]
                p = model(Xt[j].to(dev), widt[j].to(dev), samp, stat)
                out.append(p.cpu().numpy())
        return np.concatenate(out)

    # ---------- rotating-year CV over non-test years ----------
    nontest_years = sorted(set(year[~is_test].tolist()))
    folds = args.folds
    # contiguous blocks of years as val, rest as train
    blocks = np.array_split(nontest_years, folds)

    with stage(f"Rotating-year CV: {folds} folds over {len(nontest_years)} years"):
        fold_skl, fold_acc = [], []
        for fi, val_years in enumerate(blocks):
            val_years = set(val_years.tolist())
            va_i = np.isin(year, list(val_years)) & ~is_test
            tr_i = ~np.isin(year, list(val_years)) & ~is_test

            # per-fold anomalisation (train years of THIS fold only)
            Xa, ya, _ = anomalise_fold(X, y, doy, tr_i)
            model, vloss, eps = train_one(np.where(tr_i)[0], np.where(va_i)[0],
                                          Xa, ya, args.epochs, f"fold{fi}")
            p = predict(model, np.where(va_i)[0], Xa)
            t = ya[va_i]
            fin = np.isfinite(y[va_i]) & mask[None]      # mask from RAW y
            skill, acc = skill_acc(p, t, fin)
            ms = float(np.nanmean(skill[mask]))
            ma = float(np.nanmean(acc[mask]))
            fold_skl.append(ms)
            fold_acc.append(ma)
            print(f"    fold {fi} val {sorted(val_years)}: "
                  f"skill {ms:+.3f} | ACC {ma:.3f} | {eps} ep | vloss {vloss:.3f}",
                  flush=True)

        print(f"\n    CV skill {np.mean(fold_skl):+.3f} +/- {np.std(fold_skl):.3f}"
              f"   CV ACC {np.mean(fold_acc):.3f} +/- {np.std(fold_acc):.3f}")

    # ---------- final model: train on ALL non-test, evaluate on test ----------
    with stage("Final model on all non-test years -> test"):
        tr_i = ~is_test
        # small val slice just for early stopping (last 2 non-test years)
        es_years = set(nontest_years[-2:])
        va_i = np.isin(year, list(es_years)) & ~is_test
        fit_i = tr_i & ~va_i

        Xa, ya, clim_y = anomalise_fold(X, y, doy, fit_i)
        model, _, eps = train_one(np.where(fit_i)[0], np.where(va_i)[0],
                                  Xa, ya, args.epochs, "final")

        te_i = np.where(is_test)[0]
        p = predict(model, te_i, Xa)
        t = ya[is_test]
        fin = np.isfinite(y[is_test]) & mask[None]

        # overall + per-window test skill
        import xarray as xr
        wid_te = wid[is_test]
        data_vars = {}
        print(f"    trained {eps} ep")
        for w in range(n_win):
            sub_m = wid_te == w
            if sub_m.sum() == 0:
                continue
            sk, ac = skill_acc(p[sub_m], t[sub_m], fin[sub_m])
            wn = str(z["window_names"][w])
            data_vars[f"{wn}_skill"] = (("lat", "lon"), sk)
            data_vars[f"{wn}_acc"] = (("lat", "lon"), ac)
            print(f"    test {wn:>8}: skill {np.nanmean(sk[mask]):+.3f} | "
                  f"ACC {np.nanmean(ac[mask]):.3f} | "
                  f"{100*np.nanmean(sk[mask] > 0):.0f}% cells+")

        out = xr.Dataset(data_vars, coords={"lat": flat_lat, "lon": flat_lon})
        out.attrs["windows"] = ", ".join(f"{n}:{lo}-{hi}" for n, lo, hi in WINDOWS)
        out.attrs["note"] = ("anomaly-space skill = 1 - rmse/rmse_clim (clim = "
                             "zero anomaly); NaN outside IMD mask / degenerate cells")
        out.to_netcdf(out_maps)
        torch.save({"state": model.state_dict(), "args": vars(args)},
                   out_maps.replace(".nc", ".pt"))
        print(f"    -> {out_maps} (+ .pt). Per-window skill maps; open in Panoply.")


# ======================================================================

# Jupyter Notebook equivalent of command-line args
class Args:
    cmd = 'prepare' # Change to 'train' to train the model
    epochs = 60
    batch = 16
    base = 24
    drop = 0.2
    wd = 1e-3
    lr = 2e-4
    patience = 10
    folds = 5

args = Args()

if args.cmd == 'prepare':
    ecmwf_ds = ds_ecmv   # noqa: F821  <- replace with your open datasets
    imd_ds = ds_imd      # noqa: F821
    prepare(ecmwf_ds, imd_ds, CACHE)
else:
    if not os.path.exists(CACHE):
        raise SystemExit(f'run `prepare` first ({CACHE} missing)')
    build_and_run(args, CACHE, OUT_MAPS)


[ ] Coarse subset over padded India box ...
    step is int64 (max 42) -> units='D'
    {'time': 3720, 'step': 43, 'lat': 33, 'lon': 35} x 21 vars
[x] Coarse subset over padded India box  (0.0s)
[ ] Window week2 (days 8-14) ...
[########################################] | 100% Completed | 6.84 sms
[########################################] | 100% Completed | 1.48 sms
    +3720 samples
[x] Window week2 (days 8-14)  (10.7s)
[ ] Window week3_4 (days 15-28) ...


/var/folders/h4/hvzvbp993dx41cy379v1lblm0000gn/T/ipykernel_19708/326527727.py:166: RuntimeWarning: Mean of empty slice
  ya = np.nanmean(y_all.reshape(vt.shape + y_all.shape[1:]), axis=1)


[########################################] | 100% Completed | 6.75 sms
[########################################] | 100% Completed | 2.57 sms
    +3720 samples
[x] Window week3_4 (days 15-28)  (15.4s)
[ ] Window week5_6 (days 29-42) ...


/var/folders/h4/hvzvbp993dx41cy379v1lblm0000gn/T/ipykernel_19708/326527727.py:166: RuntimeWarning: Mean of empty slice
  ya = np.nanmean(y_all.reshape(vt.shape + y_all.shape[1:]), axis=1)


[########################################] | 100% Completed | 5.98 sms
[########################################] | 100% Completed | 2.53 sms
    +3720 samples
[x] Window week5_6 (days 29-42)  (14.2s)


/var/folders/h4/hvzvbp993dx41cy379v1lblm0000gn/T/ipykernel_19708/326527727.py:166: RuntimeWarning: Mean of empty slice
  ya = np.nanmean(y_all.reshape(vt.shape + y_all.shape[1:]), axis=1)



    total 11160 samples (3720 inits x 3 windows)
[ ] Strict mask + test-year holdout ...
    strict mask: 4964 cells (loose 4964)
    test years (held out of all folds): [np.int64(2022), np.int64(2023), np.int64(2024)]
[x] Strict mask + test-year holdout  (0.1s)
[ ] Caching ...
    1.06 GB -> unet_cache_mw.npz
    (raw windowed values cached; climatology/standardisation
     done per-fold at train time so each fold is leak-free)
[x] Caching  (28.2s)


In [8]:
"""
Multi-window, lead-conditioned UNet downscaling: ECMWF S2S -> IMD 0.25 rain
anomaly over India, trained jointly across several DISJOINT forecast windows,
evaluated by rotating-year cross-validation.

WHY MULTI-WINDOW (the point of this rewrite)
--------------------------------------------
The single-window version reduced each init to one sample (3720 total) and
overfit hard -- and changing model size did nothing, which means the binding
constraint is SAMPLES, not capacity. Two facts drive the fix:

  * Adding all daily leads does NOT add real data. Day-20 and day-21 from one
    init share nearly identical predictor fields and nearly identical targets.
    43 daily leads from an init is ~3 effectively-independent samples, not 43.
    Worse, adjacent leads split across train/val is near-duplication = leak.

  * Adding DISJOINT windows DOES add data. Week-2, weeks 3-4 and weeks 5-6
    dynamics genuinely differ, and their valid-date ranges don't overlap, so
    no leak. One model correcting all three, told which window it is via an
    embedding channel, shares structure across leads -- the data-rich short
    leads regularise the data-poor long leads.

So: WINDOWS below are non-overlapping. Each (init, window) is one sample with
a window-id. Effective sample count ~3x. This is the lever that actually
moves val skill; base/dropout/weight-decay are second-order.

WHY ROTATING-YEAR CV
--------------------
20 years is few. A single 3-year val holdout is noisy -- one El Nino year in
val can dominate the estimate. Rotating leave-3-out CV over the non-test years
gives a stable mean +/- spread and uses every year for validation once.
Test years are held out of ALL folds, always.

Everything else carries over from the settled pipeline: anomaly space with
train-only climatology, strict mask, mask+lat/lon static channels,
coarse-input with in-model grid_sample upsampling, GroupNorm, masked loss.

USAGE
-----
  python s2s_unet_mw.py prepare        # archive -> cached arrays (slow, once)
  python s2s_unet_mw.py train          # rotating-year CV, then final model
  python s2s_unet_mw.py train --epochs 60 --batch 16 --base 24 --folds 5
"""

import argparse
import os
import time
from contextlib import contextmanager

import numpy as np

# ---- CONFIG ----
IMD_TARGET_VAR = "rain"

# DISJOINT windows: (name, first_lead_day, last_lead_day) inclusive.
# Non-overlap in valid dates is what keeps the extra samples honest.
WINDOWS = [
    ("week1", 1, 7),
    ("week2", 8, 14),
    ("week3", 15, 21),
    ("week4", 22, 28),
    ("week5", 29, 35),
    ("week6", 36, 42),
]

CLIM_WINDOW_DAYS = 7
COARSE_PAD = 3.0
MONTHS = None                   # e.g. (6, 7, 8, 9) for JJAS inits
TEST_YEARS_N = 3                # held out of every CV fold
CACHE = "../data/cache/unet_cache_mw.npz"
OUT_MAPS = "../results/models/unet_skill_maps_mw.nc"
DTYPE = np.float32


@contextmanager
def stage(name):
    print(f"[ ] {name} ...", flush=True)
    t0 = time.perf_counter()
    yield
    print(f"[x] {name}  ({time.perf_counter() - t0:.1f}s)", flush=True)


# ======================================================================
# PREPARE  (numpy/xarray only; torch not needed here)
# ======================================================================

def normalize_step(ds, verbose=True):
    v = ds["step"].values
    if np.issubdtype(v.dtype, np.timedelta64):
        return ds
    mx = int(np.nanmax(v))
    u = "D" if mx <= 60 else ("h" if mx <= 24 * 60 else "s")
    if verbose:
        print(f"    step is {v.dtype} (max {mx}) -> units='{u}'")
    td = v.astype(np.int64).astype(f"timedelta64[{u}]").astype("timedelta64[ns]")
    return ds.assign_coords(step=("step", td))


def ensure_valid_time(ds):
    ds = normalize_step(ds)
    expected = ds["time"] + ds["step"]
    if "valid_time" in ds.coords:
        got = ds["valid_time"]
        if got.shape == expected.shape and (got.values == expected.values).all():
            return ds
        print("    WARNING: existing valid_time != time + step -> rebuilding")
        ds = ds.drop_vars("valid_time")
    return ds.assign_coords(valid_time=expected)


def _doy_matrix(doys, window, n_doy=366):
    centers = np.arange(1, n_doy + 1)
    d = np.abs(doys[None, :].astype(int) - centers[:, None])
    return (np.minimum(d, n_doy - d) <= window).astype(DTYPE)


def _clim_grid(values, doys, window):
    """(n, ...) -> (366, ...), NaN-aware DOY climatology via matmul."""
    shp = values.shape[1:]
    v2 = values.reshape(len(values), -1)
    M = _doy_matrix(doys, window)
    counts = M @ np.isfinite(v2).astype(DTYPE)
    sums = M @ np.nan_to_num(v2).astype(DTYPE)
    with np.errstate(invalid="ignore", divide="ignore"):
        clim = np.where(counts > 0, sums / np.maximum(counts, 1), np.nan)
    return clim.reshape((366,) + shp).astype(DTYPE)


def prepare(ecmwf_ds, imd_ds, cache_path):
    import xarray as xr
    from dask.diagnostics import ProgressBar

    with stage("Coarse subset over padded India box"):
        ecmwf_ds = ensure_valid_time(ecmwf_ds)
        for c in ("lat", "lon"):
            if ecmwf_ds[c].values[0] > ecmwf_ds[c].values[-1]:
                ecmwf_ds = ecmwf_ds.sortby(c)
        la0, la1 = float(imd_ds.lat.min()), float(imd_ds.lat.max())
        lo0, lo1 = float(imd_ds.lon.min()), float(imd_ds.lon.max())
        sub = ecmwf_ds.sel(lat=slice(la0 - COARSE_PAD, la1 + COARSE_PAD),
                           lon=slice(lo0 - COARSE_PAD, lo1 + COARSE_PAD))
        if MONTHS is not None:
            sub = sub.sel(time=sub["time"].dt.month.isin(list(MONTHS)))
        feature_vars = list(sub.data_vars)
        clat, clon = sub["lat"].values, sub["lon"].values
        leads = (sub["step"].values / np.timedelta64(1, "D")).astype(int)
        init = sub["time"].values
        print(f"    {dict(sub.sizes)} x {len(feature_vars)} vars")

    imd = imd_ds[IMD_TARGET_VAR]
    imd = imd.assign_coords(time=imd["time"].dt.floor("D"))
    flat_lat, flat_lon = imd["lat"].values, imd["lon"].values

    # Build each window's (X, y, doy) then stack, tagging window id.
    X_list, y_list, doy_list, wid_list = [], [], [], []
    for wid, (wname, lo, hi) in enumerate(WINDOWS):
        with stage(f"Window {wname} (days {lo}-{hi})"):
            sel = np.where((leads >= lo) & (leads <= hi))[0]
            if len(sel) == 0:
                raise ValueError(f"no leads in [{lo},{hi}]")

            with ProgressBar():
                Xw = sub.isel(step=sel).mean(dim="step").compute()
            Xa = np.stack([Xw[v].values for v in feature_vars], axis=1).astype(DTYPE)

            vt = sub["valid_time"].isel(step=sel).dt.floor("D").values
            with ProgressBar():
                y_all = imd.reindex(time=vt.ravel()).astype(DTYPE).compute().values
            ya = np.nanmean(y_all.reshape(vt.shape + y_all.shape[1:]), axis=1)

            centre = init + np.timedelta64((lo + hi) // 2, "D")
            doya = xr.DataArray(centre, dims="t").dt.dayofyear.values

            X_list.append(Xa)
            y_list.append(ya)
            doy_list.append(doya)
            wid_list.append(np.full(len(Xa), wid, dtype=np.int64))
            print(f"    +{len(Xa)} samples")

    X = np.concatenate(X_list)          # (N, V, h, w)  N = n_init * n_window
    y = np.concatenate(y_list)          # (N, H, W)
    doy = np.concatenate(doy_list)      # (N,)
    wid = np.concatenate(wid_list)      # (N,)
    year = np.concatenate([init.astype("datetime64[Y]").astype(int) + 1970] * len(WINDOWS))
    print(f"\n    total {len(X)} samples ({len(init)} inits x {len(WINDOWS)} windows)")

    with stage("Strict mask + test-year holdout"):
        # strict mask: valid on every day, per window then intersect
        mask = np.isfinite(y).all(axis=0)
        print(f"    strict mask: {int(mask.sum())} cells "
              f"(loose {int(np.isfinite(y).any(axis=0).sum())})")
        uy = np.unique(year)
        test_years = set(uy[-TEST_YEARS_N:])
        is_test = np.isin(year, list(test_years))
        print(f"    test years (held out of all folds): {sorted(test_years)}")

    with stage("Caching"):
        # NOTE: climatology + standardisation are deferred to train time,
        # because they must be recomputed per CV FOLD (train-year stats only).
        # Caching raw windowed anomable inputs would bake in a fixed split.
        np.savez_compressed(
            cache_path,
            X=X, y=y, doy=doy, wid=wid, year=year,
            mask=mask, is_test=is_test,
            clat=clat, clon=clon, flat_lat=flat_lat, flat_lon=flat_lon,
            feature_vars=np.array(feature_vars),
            window_names=np.array([w[0] for w in WINDOWS]),
        )
        print(f"    {os.path.getsize(cache_path)/1e9:.2f} GB -> {cache_path}")
        print("    (raw windowed values cached; climatology/standardisation")
        print("     done per-fold at train time so each fold is leak-free)")


# ======================================================================
# FOLD-LOCAL PREPROCESSING  (must run per fold: train-year stats only)
# ======================================================================

def anomalise_fold(X, y, doy, tr):
    """De-climatologise both sides using TRAIN-fold years only, then
    standardise predictors on train stats. Returns processed copies.

    This is the leakage-critical step. Doing it once globally would let the
    val/test years inform the climatology; doing it per fold keeps each
    fold's estimate honest.
    """
    clim_y = _clim_grid(y[tr], doy[tr], CLIM_WINDOW_DAYS)      # (366, H, W)
    clim_X = _clim_grid(X[tr], doy[tr], CLIM_WINDOW_DAYS)      # (366, V, h, w)

    ya = y - clim_y[doy - 1]
    Xa = X - clim_X[doy - 1]

    xm = np.nanmean(Xa[tr], axis=(0, 2, 3), keepdims=True)
    xs = np.nanstd(Xa[tr], axis=(0, 2, 3), keepdims=True)
    xs = np.where(xs < 1e-8, 1.0, xs)
    Xa = np.nan_to_num((Xa - xm) / xs)
    return Xa.astype(DTYPE), ya.astype(DTYPE), clim_y


# ======================================================================
# MODEL + TRAIN  (torch, lazy import)
# ======================================================================

def build_and_run(args, cache_path, out_maps):
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torch.utils.data import DataLoader, TensorDataset

    dev = ("mps" if torch.backends.mps.is_available()
           else "cuda" if torch.cuda.is_available() else "cpu")
    print(f"    device: {dev}")

    z = np.load(cache_path, allow_pickle=False)
    X, y, doy, wid = z["X"], z["y"], z["doy"], z["wid"]
    year, mask, is_test = z["year"], z["mask"], z["is_test"]
    clat, clon = z["clat"], z["clon"]
    flat_lat, flat_lon = z["flat_lat"], z["flat_lon"]
    n_win = len(z["window_names"])
    H, W = len(flat_lat), len(flat_lon)
    n_var = X.shape[1]

    # ---- coarse->fine grid_sample grid (coordinate-aware bilinear) ----
    gy = 2 * (flat_lat - clat[0]) / (clat[-1] - clat[0]) - 1
    gx = 2 * (flat_lon - clon[0]) / (clon[-1] - clon[0]) - 1
    gyy, gxx = np.meshgrid(gy, gx, indexing="ij")
    samp_np = np.stack([gxx, gyy], -1).astype(np.float32)[None]
    assert np.abs(samp_np).max() <= 1.0, "fine grid outside coarse box; raise COARSE_PAD"

    lat2 = (flat_lat[:, None] - flat_lat.mean()) / flat_lat.std()
    lon2 = (flat_lon[None, :] - flat_lon.mean()) / flat_lon.std()
    static_np = np.stack([mask.astype(DTYPE),
                          np.broadcast_to(lat2, (H, W)).astype(DTYPE),
                          np.broadcast_to(lon2, (H, W)).astype(DTYPE)])[None]

    def gn(c):
        for g in (8, 4, 2, 1):
            if c % g == 0:
                return nn.GroupNorm(g, c)

    class Block(nn.Module):
        def __init__(self, ci, co, drop=0.0):
            super().__init__()
            layers = [nn.Conv2d(ci, co, 3, padding=1), gn(co), nn.SiLU(),
                      nn.Conv2d(co, co, 3, padding=1), gn(co), nn.SiLU()]
            if drop > 0:
                layers.append(nn.Dropout2d(drop))
            self.f = nn.Sequential(*layers)

        def forward(self, x):
            return self.f(x)

    class MWUNet(nn.Module):
        """Coarse predictors + window-id -> fine anomaly field.

        The window id is embedded and broadcast as an extra coarse-input
        channel, so one model corrects all windows and shares structure
        across leads. Everything else is the settled downscaling UNet:
        coarse encoder -> grid_sample to fine -> static channels -> small
        UNet refinement.
        """
        def __init__(self, n_var, n_win, base=24, drop=0.2, emb=4):
            super().__init__()
            self.emb = nn.Embedding(n_win, emb)
            self.enc_c1 = Block(n_var + emb, base * 2)
            self.enc_c2 = Block(base * 2, base * 2)
            self.inp = Block(base * 2 + 3, base)
            self.d1 = Block(base, base * 2, drop)
            self.d2 = Block(base * 2, base * 4, drop)
            self.bott = Block(base * 4, base * 4, drop)
            self.u2 = nn.ConvTranspose2d(base * 4, base * 2, 2, stride=2)
            self.du2 = Block(base * 4, base * 2, drop)
            self.u1 = nn.ConvTranspose2d(base * 2, base, 2, stride=2)
            self.du1 = Block(base * 2, base)
            self.head = nn.Conv2d(base, 1, 1)
            self.pool = nn.MaxPool2d(2)

        def forward(self, xc, wid, samp, static):
            b, _, h, w = xc.shape
            e = self.emb(wid)[:, :, None, None].expand(-1, -1, h, w)
            xc = torch.cat([xc, e], dim=1)
            c = self.enc_c2(self.enc_c1(xc))
            f = F.grid_sample(c, samp.expand(b, -1, -1, -1),
                              mode="bilinear", align_corners=True)
            f = torch.cat([f, static.expand(b, -1, -1, -1)], dim=1)
            H0, W0 = f.shape[-2:]
            f = F.pad(f, (0, (-W0) % 4, 0, (-H0) % 4), mode="replicate")
            e0 = self.inp(f)
            e1 = self.d1(self.pool(e0))
            e2 = self.d2(self.pool(e1))
            u = self.du2(torch.cat([self.u2(self.bott(e2)), e1], dim=1))
            u = self.du1(torch.cat([self.u1(u), e0], dim=1))
            return self.head(u)[:, :, :H0, :W0].squeeze(1)

    samp = torch.tensor(samp_np).to(dev)
    stat = torch.tensor(static_np).to(dev)

    def masked_mse(pred, target, m):
        se = (pred - target) ** 2 * m
        return se.sum() / m.sum().clamp(min=1.0)

    def skill_acc(p, t, fin):
        """Per-cell skill (vs zero-anomaly clim) and ACC over masked cells."""
        se_m = np.where(fin, (t - p) ** 2, np.nan)
        se_c = np.where(fin, t ** 2, np.nan)
        with np.errstate(invalid="ignore"):
            rmse_m = np.sqrt(np.nanmean(se_m, axis=0))
            rmse_c = np.sqrt(np.nanmean(se_c, axis=0))
            ok = rmse_c > 1e-6                       # guard degenerate cells (-inf fix)
            skill = np.where(ok, 1 - rmse_m / np.where(ok, rmse_c, 1), np.nan)
            tm = np.nanmean(np.where(fin, t, np.nan), axis=0)
            pm = np.nanmean(np.where(fin, p, np.nan), axis=0)
            num = np.nansum(np.where(fin, (t - tm) * (p - pm), np.nan), axis=0)
            den = np.sqrt(np.nansum(np.where(fin, (t - tm) ** 2, np.nan), axis=0)
                          * np.nansum(np.where(fin, (p - pm) ** 2, np.nan), axis=0))
            acc = np.where(den > 0, num / den, np.nan)
        return skill, acc

    def train_one(tr_idx, va_idx, Xa, ya, max_epochs, tag):
        Xt = torch.tensor(Xa)
        yt = torch.tensor(np.nan_to_num(ya))
        widt = torch.tensor(wid)
        fin = torch.tensor((np.isfinite(ya) & mask[None]).astype(DTYPE))

        def dl(idx, sh):
            return DataLoader(TensorDataset(Xt[idx], widt[idx], yt[idx], fin[idx]),
                              batch_size=args.batch, shuffle=sh)

        tr_dl, va_dl = dl(tr_idx, True), dl(va_idx, False)
        model = MWUNet(n_var, n_win, base=args.base, drop=args.drop).to(dev)
        opt = torch.optim.AdamW(model.parameters(), lr=args.lr,
                                weight_decay=args.wd)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max_epochs)

        best, best_state, wait = np.inf, None, 0
        for ep in range(max_epochs):
            model.train()
            for xb, wb, yb, mb in tr_dl:
                xb, wb, yb, mb = [t.to(dev) for t in (xb, wb, yb, mb)]
                loss = masked_mse(model(xb, wb, samp, stat), yb, mb)
                opt.zero_grad(set_to_none=True)
                loss.backward()
                opt.step()
            sched.step()

            model.eval()
            vl, n = 0.0, 0
            with torch.no_grad():
                for xb, wb, yb, mb in va_dl:
                    xb, wb, yb, mb = [t.to(dev) for t in (xb, wb, yb, mb)]
                    vl += float(masked_mse(model(xb, wb, samp, stat), yb, mb)) * len(xb)
                    n += len(xb)
            vl /= n
            if vl < best - 1e-6:
                best, wait = vl, 0
                best_state = {k: v.detach().cpu().clone()
                              for k, v in model.state_dict().items()}
            else:
                wait += 1
            if wait >= args.patience:
                break
        model.load_state_dict(best_state)
        return model, best, ep + 1

    def predict(model, idx, Xa):
        Xt = torch.tensor(Xa)
        widt = torch.tensor(wid)
        out = []
        model.eval()
        with torch.no_grad():
            for a in range(0, len(idx), args.batch):
                j = idx[a:a + args.batch]
                p = model(Xt[j].to(dev), widt[j].to(dev), samp, stat)
                out.append(p.cpu().numpy())
        return np.concatenate(out)

    # ---------- rotating-year CV over non-test years ----------
    nontest_years = sorted(set(year[~is_test].tolist()))
    folds = args.folds
    # contiguous blocks of years as val, rest as train
    blocks = np.array_split(nontest_years, folds)

    with stage(f"Rotating-year CV: {folds} folds over {len(nontest_years)} years"):
        fold_skl, fold_acc = [], []
        for fi, val_years in enumerate(blocks):
            val_years = set(val_years.tolist())
            va_i = np.isin(year, list(val_years)) & ~is_test
            tr_i = ~np.isin(year, list(val_years)) & ~is_test

            # per-fold anomalisation (train years of THIS fold only)
            Xa, ya, _ = anomalise_fold(X, y, doy, tr_i)
            model, vloss, eps = train_one(np.where(tr_i)[0], np.where(va_i)[0],
                                          Xa, ya, args.epochs, f"fold{fi}")
            p = predict(model, np.where(va_i)[0], Xa)
            t = ya[va_i]
            fin = np.isfinite(y[va_i]) & mask[None]      # mask from RAW y
            skill, acc = skill_acc(p, t, fin)
            ms = float(np.nanmean(skill[mask]))
            ma = float(np.nanmean(acc[mask]))
            fold_skl.append(ms)
            fold_acc.append(ma)
            print(f"    fold {fi} val {sorted(val_years)}: "
                  f"skill {ms:+.3f} | ACC {ma:.3f} | {eps} ep | vloss {vloss:.3f}",
                  flush=True)

        print(f"\n    CV skill {np.mean(fold_skl):+.3f} +/- {np.std(fold_skl):.3f}"
              f"   CV ACC {np.mean(fold_acc):.3f} +/- {np.std(fold_acc):.3f}")

    # ---------- final model: train on ALL non-test, evaluate on test ----------
    with stage("Final model on all non-test years -> test"):
        tr_i = ~is_test
        # small val slice just for early stopping (last 2 non-test years)
        es_years = set(nontest_years[-2:])
        va_i = np.isin(year, list(es_years)) & ~is_test
        fit_i = tr_i & ~va_i

        Xa, ya, clim_y = anomalise_fold(X, y, doy, fit_i)
        model, _, eps = train_one(np.where(fit_i)[0], np.where(va_i)[0],
                                  Xa, ya, args.epochs, "final")

        te_i = np.where(is_test)[0]
        p = predict(model, te_i, Xa)
        t = ya[is_test]
        fin = np.isfinite(y[is_test]) & mask[None]

        # overall + per-window test skill
        import xarray as xr
        wid_te = wid[is_test]
        data_vars = {}
        print(f"    trained {eps} ep")
        for w in range(n_win):
            sub_m = wid_te == w
            if sub_m.sum() == 0:
                continue
            sk, ac = skill_acc(p[sub_m], t[sub_m], fin[sub_m])
            wn = str(z["window_names"][w])
            data_vars[f"{wn}_skill"] = (("lat", "lon"), sk)
            data_vars[f"{wn}_acc"] = (("lat", "lon"), ac)
            print(f"    test {wn:>8}: skill {np.nanmean(sk[mask]):+.3f} | "
                  f"ACC {np.nanmean(ac[mask]):.3f} | "
                  f"{100*np.nanmean(sk[mask] > 0):.0f}% cells+")

        out = xr.Dataset(data_vars, coords={"lat": flat_lat, "lon": flat_lon})
        out.attrs["windows"] = ", ".join(f"{n}:{lo}-{hi}" for n, lo, hi in WINDOWS)
        out.attrs["note"] = ("anomaly-space skill = 1 - rmse/rmse_clim (clim = "
                             "zero anomaly); NaN outside IMD mask / degenerate cells")
        out.to_netcdf(out_maps)
        torch.save({"state": model.state_dict(), "args": vars(args)},
                   out_maps.replace(".nc", ".pt"))
        print(f"    -> {out_maps} (+ .pt). Per-window skill maps; open in Panoply.")


# ======================================================================

# Jupyter Notebook equivalent of command-line args
class Args:
    cmd = 'train' # Change to 'train' to train the model
    epochs = 60
    batch = 16
    base = 24
    drop = 0.2
    wd = 1e-3
    lr = 2e-4
    patience = 10
    folds = 5

args = Args()

if args.cmd == 'prepare':
    ecmwf_ds = ds_ecmv   # noqa: F821  <- replace with your open datasets
    imd_ds = ds_imd      # noqa: F821
    prepare(ecmwf_ds, imd_ds, CACHE)
else:
    if not os.path.exists(CACHE):
        raise SystemExit(f'run `prepare` first ({CACHE} missing)')
    build_and_run(args, CACHE, OUT_MAPS)


    device: mps
[ ] Rotating-year CV: 5 folds over 17 years ...


/var/folders/h4/hvzvbp993dx41cy379v1lblm0000gn/T/ipykernel_19708/3442988177.py:343: RuntimeWarning: Mean of empty slice
  rmse_m = np.sqrt(np.nanmean(se_m, axis=0))
/var/folders/h4/hvzvbp993dx41cy379v1lblm0000gn/T/ipykernel_19708/3442988177.py:344: RuntimeWarning: Mean of empty slice
  rmse_c = np.sqrt(np.nanmean(se_c, axis=0))
/var/folders/h4/hvzvbp993dx41cy379v1lblm0000gn/T/ipykernel_19708/3442988177.py:347: RuntimeWarning: Mean of empty slice
  tm = np.nanmean(np.where(fin, t, np.nan), axis=0)
/var/folders/h4/hvzvbp993dx41cy379v1lblm0000gn/T/ipykernel_19708/3442988177.py:348: RuntimeWarning: Mean of empty slice
  pm = np.nanmean(np.where(fin, p, np.nan), axis=0)


    fold 0 val [2005, 2006, 2007, 2008]: skill +0.025 | ACC 0.246 | 19 ep | vloss 25.319
    fold 1 val [2009, 2010, 2011, 2012]: skill +0.030 | ACC 0.242 | 18 ep | vloss 18.954
    fold 2 val [2013, 2014, 2015]: skill +0.029 | ACC 0.260 | 18 ep | vloss 19.406
    fold 3 val [2016, 2017, 2018]: skill +0.029 | ACC 0.260 | 17 ep | vloss 18.145


KeyboardInterrupt: 